In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom.
# When running headlessly (CI, HPC), uncomment the ``matplotlib.use('Agg')``
# line *before* the pyplot import so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, zscore

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from independent_vector_analysis import iva_g  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    analyzers_to_datasets,
    compute_wavelet_datasets,
    load_analyzers,
    participant_labels,
)
from src.analysis import iva_quality  # noqa: E402
from src.analysis.pca_polarity import (  # noqa: E402
    align_pc1_signs,
    apply_pc1_signs,
    topography_consistency,
)
from src.analysis.wavelet_ica import (  # noqa: E402
    align_iva_component_signs,
    iva_component_patterns,
    normalize_patterns_per_subject,
    zscore_by_time,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
)
from src.visualization.iva_quality_plots import (  # noqa: E402
    plot_participant_topomaps,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
%matplotlib inline
print("Setup complete.")

# IVA Channel Decomposition — Quality via Topomap & Time Correlations

This notebook **extends** the channel-as-independent IVA workflow
([`wavelet_iva_channel.ipynb`](wavelet_iva_channel.ipynb)) with a **quality
read-out** of the decomposition. It runs the same IVA-G pipeline (z-score →
per-subject channel PCA → IVA-G → sign alignment) to recover, per subject `s`
and per component `k`:

- a **channel topography** `iva_components[s, k]` — shape `(C,)`
- a **shared spectro-temporal source** `iva_sources[s, k]` — shape `(F, T)`

It then scores every `(subject, component)` pair against two **references** and
shows the two scores on a scatterplot.

## Reference 1 — Topomap (x-axis)

The reference topography is the **group-averaged first-principal-component map
of the raw evoked response**, exactly as computed in
[`00-preprocessing/assr_raw_pca_analysis.ipynb`](../00-preprocessing/assr_raw_pca_analysis.ipynb):
per subject, epoch the raw voltage around every stimulus onset, average over
trials, z-score each channel, take the **first channel-PCA loading** (the
evoked topography), align polarity across subjects and average. We correlate
this reference topomap with **every** subject's **every** IVA-component
topography.

## Reference 2 — Time (y-axis): onset-locked, rigid 500 ms boxcar

The goal is the component reflecting the **low-level (early) onset response** to
the stimulus. The ASSR is delivered as **continuous 40 Hz stimulation**, so a
*sustained* response does not vary in time and is invisible under `zscore_by_time`
(this is why an on/off indicator over the whole recording gave correlations ≈ 0).
What *is* time-locked is the **transient right after each `fam+` onset**, so we
build the reference around the onset:

1. **Onset-lock & average.** Reduce each IVA source `(F, T)` to a `(T,)` time
   course two ways, then epoch it around every onset (`[-EPOCH_PRE_S,
   EPOCH_POST_S]`) and average → the component's onset-triggered mean response
   with a pre-onset baseline (this restores SNR and a meaningful baseline).
2. **Compare against one rigid boxcar.** The "expected response" is `0` before
   the onset, `1` for the fixed `RESP_DURATION_S` = **500 ms** stimulus window
   after it, and `0` afterwards (the OFF edge is the stimulus offset). The
   window is **not** fitted per component — every component is scored against
   the same reference, so the numbers are directly comparable. Comparison is
   **zero-lag Pearson**.

The two frequency reductions (unchanged):

- **PCA variant** — first PC over the frequency axis (per subject, per
  component) → a `(T,)` time course.
- **40 Hz variant** — the single `40 Hz` frequency row → a `(T,)` time course
  (the ASSR stimulation frequency).

## Output

Two scatterplots (topomap correlation vs. onset-locked time correlation), each
point a `(subject, component)` pair — **colour = component**:

1. **PCA frequency-reduction** variant on the y-axis.
2. **40 Hz-only** variant on the y-axis.

> Both PCA and IVA fix component signs only up to a flip. Left unhandled, this
> would turn genuine matches into negative correlations. Step 5b controls every
> flip: IVA's alignment makes each component's sign consistent **across
> subjects**, the frequency-PCA's extra per-subject flip is anchored to the
> signed source, and each component is finally oriented so its **topomap**
> correlation reads positive. A component's topography and source share one
> sign, so the time correlation then *follows* that orientation — a true
> topomap-vs-time sign disagreement (opposite quadrants) is preserved and
> meaningful, while `|r|` is invariant to all of it.

In [ ]:
# ── Experiment configuration ───────────────────────────────────
# The reference topomap is defined from the ASSR stimulus-locked evoked
# response, so this quality analysis targets the ASSR experiment.
EXPERIMENT_NAME = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO  # default per project convention
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    MUSIC_TYPES = [MusicTypeVariants.ASSR]
else:
    MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)
KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject / channel / time subset (mirror wavelet_iva_channel.ipynb) ──
N_SUBJECTS_SUBSET: int | None = 5
N_CHANNELS_SUBSET: int | None = 32   # first N channels (PCA reduces this axis)
N_TIMES_SUBSET: int | None = 3000    # first N time samples (F*T = n_freqs * this)

# ── IVA settings ──────────────────────────────────────────────
N_COMPONENTS_PCA = 15  # per-subject PCA dim over channels (= N in IVA's (N, T, K))
IVA_OPT_APPROACH = "newton"
IVA_MAX_ITER = 64
IVA_W_DIFF_STOP = 1e-6
IVA_VERBOSE = False
IVA_RANDOM_STATE = 42

# ── Stimulus-locked epoch geometry (shared by BOTH quality references) ──────
# Taken from the paradigm definition (src.definitions.constants.AssrEpoch), so
# this notebook, the headless `run_wavelet_iva_channel.py --quality` path
# (src.analysis.iva_quality) and the 00-preprocessing ASSR PCA notebooks all cut
# the SAME window:
#   0.1 s baseline before onset, then 1.0 s after onset
#   = 0.5 s stimulus + 0.5 s post-stimulus
# The post-onset length is capped by the shortest inter-onset gap wherever it is
# applied, so no epoch can reach a neighbouring stimulus.
EPOCH_PRE_S = AssrEpoch.PRE_ONSET_S    # pre-onset baseline (both references)
EPOCH_POST_S = AssrEpoch.POST_ONSET_S  # post-onset span (both references)
REF_PRE_PAD_S = EPOCH_PRE_S            # reference topomap uses the same baseline

# The rigid expected-response window for the boxcar reference is the stimulus
# length itself — the OFF edge is the stimulus offset, so it is the same number.
RESP_DURATION_S = AssrEpoch.STIMULUS_DURATION_S

# Reference channel that anchors the group polarity of the PC1 loading.
REF_POLARITY_CHANNEL = iva_quality.REF_POLARITY_CHANNEL
# Frequency (Hz) used by the "40 Hz-only" time-reduction variant.
ASSR_FREQ = iva_quality.ASSR_FREQ

# ── Which components to score ─────────────────────────────────────────
# None = every IVA component (0 .. N_COMPONENTS_PCA-1). Set to an explicit list
# of 0-based component indices to focus the scatterplot on a subset.
COMPONENTS_TO_PLOT: list[int] | None = None

# ── Plot saving ───────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "05-wavelet-iva-analysis"
    / "plots"
    / EXPERIMENT_NAME.value
    / "broadband"
    / "iva_channel_quality"
    / f"pca_{N_COMPONENTS_PCA}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment / group : {EXPERIMENT_NAME.value} — {CONDITION.value}")
print(f"Frequencies        : {FREQS[0]:.1f}-{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)")
print(f"Per-subject PCA dim : {N_COMPONENTS_PCA}  (over channels)")
print(f"Onset epoch        : [-{EPOCH_PRE_S}, {EPOCH_POST_S}] s around each onset "
      f"({AssrEpoch.STIMULUS_DURATION_S} s stimulus + "
      f"{AssrEpoch.POST_STIMULUS_S} s post-stimulus)")
print(f"Response window    : {RESP_DURATION_S * 1000:.0f} ms (rigid, onset-locked)")
print(f"Plots -> {PLOTS_DIR}")

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
    experiment_name=EXPERIMENT_NAME,
)
datasets = analyzers_to_datasets(analyzers)

# Subset subjects / channels / times (applied to ``datasets`` only — the full
# ``analyzers[...].data`` is kept intact and reused for the reference topomap).
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

In [ ]:
broadband_datasets = compute_wavelet_datasets(
    datasets=datasets,
    analyzers=analyzers,
    experiment_name=EXPERIMENT_NAME,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=(
        ProjectPaths.NOTEBOOKS_DIR
        / "03-wavelet-analysis"
        / "wavelet_cache"
        / EXPERIMENT_NAME.value
        / "broadband"
    ),
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

# MNE info for the topomaps, restricted to the IVA channel subset. This is the
# canonical channel order of ``iva_components`` below.
iva_info = analyzers[LABEL].info
iva_info = mne.pick_info(iva_info, mne.pick_types(iva_info, eeg=True))
if n_channels < len(iva_info.ch_names):
    iva_info = mne.pick_info(iva_info, list(range(n_channels)))
iva_ch_names = list(iva_info["ch_names"])

print(f"Dataset    : {LABEL}")
print(f"Shape      : {bb_data.shape}  (subjects x channels x freqs x times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}-{FREQS[-1]:.1f} Hz ({n_freqs} steps)")
print(f"IVA channels : {n_channels}  (first={iva_ch_names[0]}, last={iva_ch_names[-1]})")

# Per-subject participant labels (``019`` etc.) taken from the metadata
# sidecar written next to the concatenated array. ``participant_labels`` maps
# subject index -> CONCATENATED_PERSON_INDEX -> PARTICIPANT_ID and raises if the
# mapping is unavailable, so a wrong label can never pass silently.
try:
    subject_ids = participant_labels(analyzers[LABEL].filtered_df, n_subjects)
except ValueError as _exc:
    print(f"Participant labels unavailable ({_exc}); using subject indices.")
    subject_ids = [f"#{s:03d}" for s in range(n_subjects)]

print(f"Subject IDs  : {subject_ids}")

## Step 1 — Run the channel-as-independent IVA-G

Identical to [`wavelet_iva_channel.ipynb`](wavelet_iva_channel.ipynb): z-score
along time, per-subject **PCA over channels** (square-mixing requirement of
IVA-G), run IVA-G, resolve the per-subject sign ambiguity, then recover the
per-subject spectro-temporal **sources** `(S, N_PCA, F, T)` and channel
**topographies** `(S, N_PCA, C)`.

In [ ]:
# Step 1a — z-score along time, per-subject reshape to (S, C, F*T).
bb_z = zscore_by_time(bb_data)
n_samples_ft = n_freqs * n_times
X_subjects = bb_z.reshape(n_subjects, n_channels, n_samples_ft)

if N_COMPONENTS_PCA > n_channels:
    raise ValueError(
        f"N_COMPONENTS_PCA ({N_COMPONENTS_PCA}) must be <= n_channels "
        f"({n_channels}); PCA reduces the channel axis here."
    )

# Step 1b — per-subject PCA over channels -> IVA input (N_PCA, F*T, S).
pcas: list[PCA] = []
pca_scores_per_subject = np.zeros((n_subjects, N_COMPONENTS_PCA, n_samples_ft))
for k in range(n_subjects):
    subj_matrix = X_subjects[k].T  # (F*T, C)
    pca = PCA(n_components=N_COMPONENTS_PCA, random_state=IVA_RANDOM_STATE)
    scores = pca.fit_transform(subj_matrix)
    pcas.append(pca)
    pca_scores_per_subject[k] = scores.T
X_pca = np.ascontiguousarray(pca_scores_per_subject.transpose(1, 2, 0))

# Step 1c — IVA-G.
rng = np.random.default_rng(IVA_RANDOM_STATE)
W_init = rng.standard_normal((N_COMPONENTS_PCA, N_COMPONENTS_PCA, n_subjects))
W, cost, Sigma_N, _isi = iva_g(
    X_pca,
    opt_approach=IVA_OPT_APPROACH,
    whiten=True,
    verbose=IVA_VERBOSE,
    W_init=W_init,
    max_iter=IVA_MAX_ITER,
    W_diff_stop=IVA_W_DIFF_STOP,
)

# Step 1d — resolve per-subject sign ambiguity (aligns W across subjects).
sigma_corr, W, sign_flips = align_iva_component_signs(Sigma_N, W)

# Step 1e — recover spectro-temporal sources + channel topographies.
iva_scores_pca = np.zeros((n_subjects, N_COMPONENTS_PCA, n_samples_ft))
iva_components = np.zeros((n_subjects, N_COMPONENTS_PCA, n_channels))
for k in range(n_subjects):
    W_k = W[:, :, k]
    iva_scores_pca[k] = W_k @ X_pca[:, :, k]
    # Forward (mixing) patterns — this is what the raw-PCA reference
    # topomap is too, so the correlation below is pattern-vs-pattern.
    iva_components[k] = iva_component_patterns(
        W_k, pcas[k].components_
    )  # (N_PCA, C)
iva_sources = iva_scores_pca.reshape(n_subjects, N_COMPONENTS_PCA, n_freqs, n_times)

print(f"IVA-G iterations   : {len(cost)}  final cost = {cost[-1]:.6f}")
print(f"iva_sources        : {iva_sources.shape}  (S, N_PCA, F, T)")
print(f"iva_components      : {iva_components.shape}  (S, N_PCA, C)")

## Step 2 — Reference topomap (raw evoked channel-PCA, group mean)

Reproduces the reference from
[`00-preprocessing/assr_raw_pca_analysis.ipynb`](../00-preprocessing/assr_raw_pca_analysis.ipynb),
computed from the **full** (un-subset) raw voltage of every loaded subject:

1. z-score each channel over the whole recording (equal electrode influence),
2. epoch around every onset and **average over trials** (the evoked response),
3. fit a **PCA over channels** (time samples = observations) and keep the
   **first loading** (the evoked topography),
4. align each subject's loading polarity to a common template, anchor the
   overall sign with `REF_POLARITY_CHANNEL`, and **average across subjects**.

The **epoch is the paradigm window** — `0.1` s baseline before onset, then `1.0` s
after it (`0.5` s stimulus + `0.5` s post-stimulus) from
`src.definitions.constants.AssrEpoch`, capped by the shortest inter-onset gap.
This is the *same* window as the onset-locked time reference in Step 3, so both
quality axes are scored over identical epochs. (It previously ran to
`gaps.min()` ≈ 1.25 s, which stretched the reference past the stimulus and made
its pre-onset baseline the tail of the previous stimulus.)

Polarity alignment and its consistency diagnostic come from
`src.analysis.pca_polarity`, shared with the 00-preprocessing notebooks — template
alignment maximises cross-subject topography agreement, which is what makes the
group mean a meaningful reference. The printed diagnostic flags any subject that
still anti-correlates with the group: that is a topographic outlier rather than a
sign problem, and it is averaged into the reference.

The reference is built over all channels and indexed by channel name, then
restricted to the IVA channel subset (`iva_ch_names`) so it aligns with
`iva_components` channel-for-channel.

> With `N_CHANNELS_SUBSET` set (exploration default), the reference is shown on
> only the **first N channels**, which are spatially clustered — the topomap
> interpolation therefore looks saturated over the rest of the scalp. Set
> `N_CHANNELS_SUBSET = None` for the full-montage reference.

In [ ]:
ref_analyzer = analyzers[LABEL]
raw_v = ref_analyzer.data  # (S_full, C_full, T_full) preprocessed voltage
ref_info = mne.pick_info(
    ref_analyzer.info, mne.pick_types(ref_analyzer.info, eeg=True)
)
ref_ch_names = list(ref_info["ch_names"])
ref_onsets = ref_analyzer.stimulus_onsets
ref_sfreq = float(ref_analyzer.info["sfreq"])
assert raw_v.shape[1] == len(ref_ch_names), "raw voltage vs info channel mismatch"
assert ref_onsets is not None, "No stimulus onsets available for the reference."

# Epoch window: the paradigm span (0.1 s baseline + 1.0 s post-onset), capped by
# the shortest inter-onset gap. Identical to the onset-locked time reference in
# Step 3 and to iva_quality.reference_topomap, so both quality axes are scored on
# one and the same window.
ref_gaps = np.diff(ref_onsets)
REF_PRE = AssrEpoch.pre_onset_samples(ref_sfreq)
REF_POST = AssrEpoch.post_onset_samples(ref_sfreq, min_gap=int(ref_gaps.min()))

# Per-subject PC1 loading of the trial-averaged, z-scored evoked response.
ref_loadings = np.zeros((raw_v.shape[0], raw_v.shape[1]))
for si in range(raw_v.shape[0]):
    sig = zscore(raw_v[si], axis=1)  # unit variance per channel
    # Shared epoching helper (same implementation the headless path uses).
    evoked, _n_used = iva_quality.epoch_average(
        sig, ref_onsets, REF_PRE, REF_POST
    )  # (C, win)
    pca = PCA(n_components=1)
    pca.fit(evoked.T)  # observations = time samples, variables = channels
    ref_loadings[si] = pca.components_[0]

# Align PC1 polarity across subjects (template alignment maximises cross-subject
# topography agreement), then anchor the group's overall orientation.
ref_signs, anchor = align_pc1_signs(
    ref_loadings, channel_names=ref_ch_names, reference=REF_POLARITY_CHANNEL
)
ref_loadings = apply_pc1_signs(ref_loadings, ref_signs)
ref_consistency = topography_consistency(ref_loadings)

# Group-mean reference topomap, restricted to the IVA channel subset by name.
ref_topo_by_name = dict(zip(ref_ch_names, ref_loadings.mean(axis=0)))
ref_topo = np.array([ref_topo_by_name[ch] for ch in iva_ch_names])  # (n_channels,)

print(f"Reference built from {raw_v.shape[0]} subjects, {raw_v.shape[1]} channels.")
print(f"Epoch window            : {REF_PRE + REF_POST} samples "
      f"({REF_PRE} pre, {REF_POST} post) = "
      f"[{-REF_PRE / ref_sfreq:.3f}, {(REF_POST - 1) / ref_sfreq:.3f}] s")
print(f"Polarity anchor channel : {anchor} "
      f"({int((ref_signs < 0).sum())}/{len(ref_signs)} subject(s) flipped)")
print(f"Topographies agreeing   : {ref_consistency.n_agreeing}/"
      f"{ref_consistency.n_subjects}  "
      f"(median pairwise r={ref_consistency.median_pairwise_r:+.2f}, "
      f"weakest r={ref_consistency.min_subject_r:+.2f})")
if ref_consistency.n_agreeing < ref_consistency.n_subjects:
    print(f"  WARNING: {ref_consistency.n_subjects - ref_consistency.n_agreeing} "
          f"subject(s) anti-correlate with the group topography — a topographic "
          f"outlier, not a sign problem. The reference averages them in.")
print(f"Reference topomap (IVA subset) : {ref_topo.shape}")

# Visual check of the reference topomap on the IVA channel subset.
vlim = max(float(np.percentile(np.abs(ref_topo), 99)), 1e-12)
fig, ax = plt.subplots(figsize=(4.2, 4.0))
im, _ = mne.viz.plot_topomap(
    ref_topo, iva_info, axes=ax, show=False, cmap="RdBu_r",
    vlim=(-vlim, vlim), contours=4,
)
ax.set_title("Reference topomap\n(group raw-PCA evoked PC1)", fontsize=10)
fig.colorbar(im, ax=ax, shrink=0.7, label="PC1 loading (a.u.)")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "reference_topomap.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

## Step 3 — Onset-locked epoch window & the rigid boxcar reference

Set up the onset-triggered epoch and the single "expected response" boxcar.

The epoch is the **paradigm window**, obtained from `iva_quality.onset_window`
(which reads `src.definitions.constants.AssrEpoch`): `0.1` s baseline before
onset, then `1.0` s after onset — the `0.5` s **stimulus** plus `0.5` s of
**post-stimulus** — capped by the shortest inter-onset gap so epochs never
overlap. Step 2's reference topomap uses the identical window, and so does the
headless `run_wavelet_iva_channel.py --quality` path, so no quality metric is
scored on a different epoch than another.

The boxcar is `0` before the onset, `1` for the rigid `RESP_DURATION_S`
(500 ms = the stimulus length, so the OFF edge is the stimulus offset), and `0`
afterwards. Keeping `0.5` s of epoch *after* that edge is what lets the score
distinguish a response that stops with the stimulus from one that runs on: with
a window that ended at the stimulus offset, both would look identical. The
`0`-tail is scored, not padding.

The window is clamped to the epoch's post-onset span so it always fits, and the
cell warns if the clamp ever engages. It is used in Step 5 to score every
component against one and the same reference.

In [ ]:
_onset_samples = analyzers[LABEL].stimulus_onsets
onsets_in = _onset_samples[_onset_samples < n_times].astype(int)

# Same paradigm window as the reference topomap (Step 2) and the headless path:
# 0.1 s baseline + 1.0 s post-onset, capped by the shortest inter-onset gap.
PRE, POST = iva_quality.onset_window(onsets_in, n_times, sfreq)
W = PRE + POST
epoch_times = np.arange(-PRE, POST) / sfreq  # (W,) seconds, t=0 at onset
stim_mask = AssrEpoch.stimulus_mask(epoch_times)  # the driven 0-0.5 s interval


def onset_average(x):
    """Onset-triggered average of a 1-D time course x(t) -> (W,)."""
    acc, n_used = None, 0
    for o in onsets_in:
        s, e = o - PRE, o + POST
        if s < 0 or e > n_times:
            continue
        seg = x[s:e]
        acc = seg.astype(float) if acc is None else acc + seg
        n_used += 1
    if n_used == 0:
        raise ValueError("No onset window fits inside the IVA time window.")
    return acc / n_used, n_used


def boxcar(d_samples):
    """Expected response over the epoch: 0 baseline, 1 for d post-onset samples."""
    b = np.zeros(W)
    b[PRE:PRE + int(d_samples)] = 1.0
    return b


# The rigid reference window: the stimulus length, clamped to the epoch's
# post-onset span so it always fits inside the epoch.
RESP_SAMPLES = iva_quality.response_duration_samples(POST, sfreq)
RESP_DUR_S = RESP_SAMPLES / sfreq
REF_BOXCAR = boxcar(RESP_SAMPLES)

print(f"Onsets used      : {len(onsets_in)}")
print(f"Epoch window     : {W} samples ({PRE} pre, {POST} post) = "
      f"[{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s")
print(f"Response window  : {RESP_DUR_S * 1000:.0f} ms ({RESP_SAMPLES} samples), rigid")
print(f"Post-stimulus    : {epoch_times[-1] - RESP_DUR_S:.3f} s after the OFF edge")
if POST < int(round(EPOCH_POST_S * sfreq)):
    print(f"  NOTE: post-onset span trimmed from {EPOCH_POST_S} s to "
          f"{POST / sfreq:.3f} s by the shortest inter-onset gap.")
if RESP_SAMPLES < int(round(RESP_DURATION_S * sfreq)):
    print(f"  WARNING: the {RESP_DURATION_S * 1000:.0f} ms stimulus window does not "
          f"fit the epoch and was clamped to {RESP_DUR_S * 1000:.0f} ms.")

# Show the reference boxcar on the epoch time axis.
fig, ax = plt.subplots(figsize=(10, 2.4))
ax.plot(epoch_times, REF_BOXCAR, lw=1.6, drawstyle="steps-post", color="C0")
ax.axvline(0.0, color="red", ls="--", lw=0.8)
ax.set_ylim(-0.1, 1.1)
ax.set_yticks([0, 1])
ax.set_xlabel("Time relative to onset (s)")
ax.set_title(
    f"Rigid {RESP_DUR_S * 1000:.0f} ms onset-locked boxcar reference "
    f"(epoch [{epoch_times[0]:.1f}, {epoch_times[-1]:.1f}] s)"
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "boxcar_reference.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

## Step 4 — Topomap correlation (x-axis)

Pearson correlation between each `(subject, component)` topography
`iva_components[s, k]` and the reference topomap `ref_topo` (both `(C,)`,
channel-aligned). Signed — see the sign caveat in the header.

In [ ]:
topo_corr = np.zeros((n_subjects, N_COMPONENTS_PCA))
for s in range(n_subjects):
    for k in range(N_COMPONENTS_PCA):
        topo_corr[s, k] = pearsonr(iva_components[s, k], ref_topo)[0]

print(f"topo_corr : {topo_corr.shape}  (subjects x components)")
print(f"  range [{topo_corr.min():+.2f}, {topo_corr.max():+.2f}]  "
      f"max |r| = {np.abs(topo_corr).max():.2f}")

## Step 5 — Onset-locked time correlation (y-axis)

For each `(subject, component)` reduce the source `(F, T)` to a `(T,)` time
course two ways, **onset-average** it (Step 3 helper), then correlate that
averaged response against the **rigid boxcar** `REF_BOXCAR` (zero-lag Pearson,
the same 500 ms reference for every subject and component):

- **PCA variant** (`time_corr_pca`) — first PC over frequency (sign-anchored to
  the signed temporal marginal so it is consistent across subjects).
- **40 Hz variant** (`time_corr_40`) — the single `ASSR_FREQ` frequency row.

`onset_avg_pca` / `onset_avg_40` keep the `(S, K, W)` onset-averages for the
diagnostic plot.

In [ ]:
f40 = int(np.argmin(np.abs(FREQS - ASSR_FREQ)))
print(f"40 Hz row : FREQS[{f40}] = {FREQS[f40]:.1f} Hz")

# Onset-triggered average of each component's reduced time course, per variant.
onset_avg_pca = np.zeros((n_subjects, N_COMPONENTS_PCA, W))
onset_avg_40 = np.zeros((n_subjects, N_COMPONENTS_PCA, W))
for s in range(n_subjects):
    for k in range(N_COMPONENTS_PCA):
        src = iva_sources[s, k]  # (F, T)
        # Variant A — first PC over frequency, sign-anchored to the signed
        # temporal marginal (the frequency-PCA's own sign is otherwise arbitrary).
        pc1_time = PCA(n_components=1, random_state=IVA_RANDOM_STATE).fit_transform(
            src.T
        )[:, 0]
        if pearsonr(pc1_time, src.mean(axis=0))[0] < 0:
            pc1_time = -pc1_time
        onset_avg_pca[s, k], _ = onset_average(pc1_time)
        # Variant B — the single 40 Hz row (sign-aligned by IVA).
        onset_avg_40[s, k], _ = onset_average(src[f40])


def boxcar_correlation(onset_avgs):
    """Zero-lag Pearson of every onset-average against the rigid boxcar.

    ``onset_avgs`` is (S, K, W). ``REF_BOXCAR`` is the same reference for every
    subject and component, so the returned (S, K) scores are directly
    comparable across components.
    """
    n_comp = onset_avgs.shape[1]
    time_corr = np.zeros((n_subjects, n_comp))
    for k in range(n_comp):
        time_corr[:, k] = np.nan_to_num(
            np.array(
                [
                    pearsonr(onset_avgs[s, k], REF_BOXCAR)[0]
                    for s in range(n_subjects)
                ]
            )
        )
    return time_corr


time_corr_pca = boxcar_correlation(onset_avg_pca)
time_corr_40 = boxcar_correlation(onset_avg_40)

for name, arr in (
    ("PCA-reduced", time_corr_pca),
    ("40 Hz", time_corr_40),
):
    print(f"time_corr [{name:11s}] range [{arr.min():+.2f}, {arr.max():+.2f}]  "
          f"max |r| = {np.abs(arr).max():.2f}")

## Step 5b — Sign-ambiguity check & per-component orientation

IVA and PCA fix component signs only up to a flip, which could turn a genuine
match into a **negative** correlation. Two flips are in play, and both are now
controlled:

- **Cross-subject flip** — `align_iva_component_signs` already oriented every
  component's sign consistently across subjects, so a component's **topography**
  and its **40 Hz source** point the same way for all subjects. Because a
  component's topography and source share one sign, the topomap and both time
  correlations flip *together*.
- **Frequency-PCA flip** — the PCA-reduced time course carried an extra
  per-subject sign, anchored above to the signed temporal marginal.

This cell **verifies** cross-subject sign consistency (how many subjects agree
with each component's mean-sign — low agreement means the pattern genuinely
differs across subjects, *not* a sign artefact), then gives each component one
**global orientation** so its **topomap** correlation reads positive. The time
sign then follows from the shared component sign (it is *not* forced positive
independently — a real topomap-vs-time sign disagreement is preserved). `|r|`
is unchanged by any of this.

In [ ]:
def _sign_agreement(corr):
    """Per-component fraction of subjects agreeing with the mean-sign. (S,K)->(K,)."""
    frac = np.zeros(corr.shape[1])
    for k in range(corr.shape[1]):
        col = corr[:, k]
        ref = np.sign(col.mean()) or 1.0
        frac[k] = float(np.mean(np.sign(col) == ref))
    return frac


for name, arr in (
    ("topomap", topo_corr),
    ("time-PCA", time_corr_pca),
    ("time-40Hz", time_corr_40),
):
    frac = _sign_agreement(arr)
    weak = [k + 1 for k in range(N_COMPONENTS_PCA) if frac[k] < 0.6]
    print(
        f"[{name:9s}] cross-subject sign agreement: mean {frac.mean() * 100:3.0f}%, "
        f"min {frac.min() * 100:3.0f}%  |  <60% agreement: {weak or 'none'}"
    )

# Global per-component orientation: one flip per component so its mean topomap
# correlation is >= 0. The same flip applies to all three measures (shared
# component sign), so it is applied to every correlation array. Idempotent.
sign_per_comp = np.sign(topo_corr.mean(axis=0))
sign_per_comp[sign_per_comp == 0] = 1.0
topo_corr = topo_corr * sign_per_comp
time_corr_pca = time_corr_pca * sign_per_comp
time_corr_40 = time_corr_40 * sign_per_comp
# Orient the onset-averages the same way so the diagnostic plot matches the
# oriented correlations (a positive-going response for a positive time score).
onset_avg_pca = onset_avg_pca * sign_per_comp[None, :, None]
onset_avg_40 = onset_avg_40 * sign_per_comp[None, :, None]
n_flipped_comp = int((sign_per_comp < 0).sum())
print(
    f"Oriented {n_flipped_comp}/{N_COMPONENTS_PCA} components so the mean "
    f"topomap correlation is >= 0 (time sign follows the shared component sign)."
)

## Step 6 — Quality scatterplots

Each point is one `(subject, component)` pair: **x** = topomap correlation,
**y** = time correlation. For each frequency-reduction variant two figures are
produced:

- **Combined** — all points overlaid, **colour = component**, no labels.
- **Per-component** — one small panel per component, points are the subjects
  **labelled by 3-digit ID**; each panel keeps that component's colour so a
  point can be located across both figures.

Both axes are **auto-scaled** to a symmetric window around zero, sized from the
largest correlation magnitude in the figure (plus a little padding) and clamped
to `[-1, 1]`. The same window is shared by the combined and per-component views.

Each component carries a **score** = the mean over subjects of
`(topomap r + time r) / 2`. It equals `1.0` only when every subject sits at the
ideal **(1, 1)** corner, so a higher score means the component's dots cluster
closer to top-right. The score is shown next to each component in the combined
legend and in each per-component panel title (it depends on the y-axis time
variant, so the two variants report different scores).

In [ ]:
# Symmetric-window padding: fraction of |r|_max added on each side (a small
# absolute floor keeps very tight clusters from touching the axes).
AXIS_PAD_FRAC = 0.15
AXIS_PAD_MIN = 0.05

if COMPONENTS_TO_PLOT is None:
    comp_indices = list(range(N_COMPONENTS_PCA))
else:
    comp_indices = list(COMPONENTS_TO_PLOT)

if len(comp_indices) <= 20:
    _comp_colors = list(plt.colormaps["tab20"].colors)
    comp_colors = {k: _comp_colors[i % 20] for i, k in enumerate(comp_indices)}
else:
    cmap = plt.colormaps["hsv"]
    comp_colors = {
        k: cmap(i / len(comp_indices)) for i, k in enumerate(comp_indices)
    }


def _axis_limit(time_corr):
    """Symmetric [-lim, lim] window from the largest |r| (topomap + time)."""
    max_abs = float(
        max(
            np.abs(topo_corr[:, comp_indices]).max(),
            np.abs(time_corr[:, comp_indices]).max(),
        )
    )
    return min(1.0, max_abs * (1.0 + AXIS_PAD_FRAC) + AXIS_PAD_MIN)


def _component_score(time_corr, k):
    """Per-component quality score: mean over subjects of (topomap + time) / 2.

    Higher is better; the score equals 1.0 exactly when every subject sits at
    the ideal top-right corner (topomap r = 1, time r = 1), so it measures how
    close a component's dots are to (1, 1).
    """
    return float(np.mean((topo_corr[:, k] + time_corr[:, k]) / 2.0))


def plot_onset_diagnostic(onset_avgs, time_corr, *, variant_name, save_name):
    """Per-component group-mean onset-average + the rigid response window.

    Shows, for every component, the across-subject mean onset-triggered response
    on the epoch time axis, with the fixed response window [0, RESP_DUR_S]
    shaded — so you can see whether the component's onset response looks like a
    stimulus response and how well it fills the scored window.
    """
    group_avg = onset_avgs.mean(axis=0)  # (K, W)
    n_comp = len(comp_indices)
    ncols = min(5, n_comp)
    nrows = int(np.ceil(n_comp / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(3.0 * ncols, 2.4 * nrows), squeeze=False, sharex=True
    )
    flat = axes.flatten()
    for panel, k in enumerate(comp_indices):
        ax = flat[panel]
        ax.plot(epoch_times, group_avg[k], color=comp_colors[k], lw=1.4)
        ax.axvline(0.0, color="red", ls="--", lw=0.8)
        ax.axvspan(0.0, RESP_DUR_S, color="0.5", alpha=0.15)
        ax.axhline(0.0, color="gray", ls=":", lw=0.5)
        ax.tick_params(labelsize=6)
        ax.set_title(
            f"IC {k + 1}  (score {_component_score(time_corr, k):+.2f})",
            fontsize=8.5, color="black", fontweight="bold",
        )
    for ax in flat[n_comp:]:
        ax.axis("off")
    fig.supxlabel("Time relative to onset (s)")
    fig.supylabel("Group-mean onset-averaged component response (a.u.)")
    fig.suptitle(
        f"Onset-locked response vs {RESP_DUR_S * 1000:.0f} ms stimulus window — "
        f"{variant_name} — {LABEL}",
        fontsize=12,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / save_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


def plot_quality_scatter(time_corr, *, variant_name, save_name):
    """Combined scatter: topomap-corr (x) vs time-corr (y), colour = component.

    All components/subjects overlaid, no per-point labels. Both axes share one
    symmetric window sized from the largest correlation magnitude present.
    """
    lim = _axis_limit(time_corr)

    fig, ax = plt.subplots(figsize=(9.0, 7.5))
    for s in range(n_subjects):
        for k in comp_indices:
            ax.scatter(
                topo_corr[s, k], time_corr[s, k], color=comp_colors[k],
                marker="o", s=70, edgecolor="black", linewidth=0.3,
                alpha=0.9, zorder=2,
            )
    ax.axhline(0.0, ls="--", lw=0.6, color="gray")
    ax.axvline(0.0, ls="--", lw=0.6, color="gray")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Topomap correlation with raw-PCA reference")
    ax.set_ylabel(f"Onset-locked time correlation ({variant_name})")
    ax.set_title(
        f"IVA channel-component quality — {variant_name} — {LABEL}\n"
        f"(colour = component; window |r| ≤ {lim:.2f})"
    )

    # Legend labels carry each component's score (closeness to the (1,1) ideal).
    comp_handles = [
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor=comp_colors[k],
               markeredgecolor="black", markersize=8,
               label=f"IC {k + 1}  ({_component_score(time_corr, k):+.2f})")
        for k in comp_indices
    ]
    leg1 = ax.legend(
        handles=comp_handles, title="Component  (score)", fontsize=7,
        loc="upper left", bbox_to_anchor=(1.01, 1.0),
        ncol=1 + (len(comp_indices) > 15),
    )
    ax.add_artist(leg1)
    fig.tight_layout()
    if SAVE_PLOTS:
        # bbox_extra_artists keeps the out-of-axes legend inside the tight crop.
        fig.savefig(
            PLOTS_DIR / save_name, dpi=150, bbox_inches="tight",
            bbox_extra_artists=(leg1,),
        )
    plt.show()
    plt.close("all")


def plot_quality_scatter_per_component(time_corr, *, variant_name, save_name):
    """One small scatter per component; points are subjects labelled by ID.

    Each panel keeps that component's colour (matching the combined plot) and
    the shared symmetric window, so a point can be located across both figures.
    """
    lim = _axis_limit(time_corr)
    n_comp = len(comp_indices)
    ncols = min(5, n_comp)
    nrows = int(np.ceil(n_comp / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(2.9 * ncols, 2.9 * nrows), squeeze=False,
    )
    flat = axes.flatten()
    for panel, k in enumerate(comp_indices):
        ax = flat[panel]
        color = comp_colors[k]
        for s in range(n_subjects):
            x, y = topo_corr[s, k], time_corr[s, k]
            ax.scatter(
                x, y, color=color, marker="o", s=55, edgecolor="black",
                linewidth=0.3, alpha=0.9, zorder=2,
            )
            ax.annotate(
                subject_ids[s], (x, y), textcoords="offset points",
                xytext=(3.0, 2.5), fontsize=6, color="black", zorder=3,
            )
        ax.axhline(0.0, ls="--", lw=0.5, color="gray")
        ax.axvline(0.0, ls="--", lw=0.5, color="gray")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect("equal", adjustable="box")
        ax.tick_params(labelsize=6)
        ax.set_title(
            f"IC {k + 1}  (score {_component_score(time_corr, k):+.2f})",
            fontsize=8.5, color="black", fontweight="bold",
        )
    for ax in flat[n_comp:]:
        ax.axis("off")
    fig.supxlabel("Topomap correlation with raw-PCA reference")
    fig.supylabel(f"Onset-locked time correlation ({variant_name})")
    fig.suptitle(
        f"Per-component quality (label = participant ID) — {variant_name} — {LABEL}"
        f"  |  window |r| ≤ {lim:.2f}",
        fontsize=12,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / save_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


plot_onset_diagnostic(
    onset_avg_pca, time_corr_pca,
    variant_name="PCA frequency-reduction",
    save_name="quality_onset_response_pca.png",
)
plot_quality_scatter(
    time_corr_pca,
    variant_name="PCA frequency-reduction",
    save_name="quality_scatter_time_pca.png",
)
plot_quality_scatter_per_component(
    time_corr_pca,
    variant_name="PCA frequency-reduction",
    save_name="quality_scatter_time_pca_per_component.png",
)

In [ ]:
plot_onset_diagnostic(
    onset_avg_40, time_corr_40,
    variant_name="40 Hz band only",
    save_name="quality_onset_response_40hz.png",
)
plot_quality_scatter(
    time_corr_40,
    variant_name="40 Hz band only",
    save_name="quality_scatter_time_40hz.png",
)
plot_quality_scatter_per_component(
    time_corr_40,
    variant_name="40 Hz band only",
    save_name="quality_scatter_time_40hz_per_component.png",
)

---
## Step 7 — One participant-comparison figure per component

The scatterplots collapse each `(subject, component)` pair to two numbers. This
step writes the underlying maps so a low topomap correlation can be *looked at*:
one figure per IVA component holding every participant's channel topography side
by side, followed by the group mean and the raw-PCA reference.

Numbering is 1-based and follows the **IVA component order**, matching the
`IC <k+1>` labels of the figures above — not the score ranking. Within a figure
every participant panel shares one colour limit so the panels are directly
comparable, and the suptitle reports that limit together with the mean / min /
max `r` across participants. Limits are *not* shared across components, whose
pattern magnitudes differ by construction; the reference keeps its own scale
because it is a PC1 loading rather than a pattern. Panels stay in subject order
so a participant sits in the same grid position for every component.

`iva_components` was not oriented in place in Step 5b (only the correlation
arrays and onset averages were), so the shared per-component sign is applied
here — otherwise a map would contradict the `r` printed beside it.

In [ ]:
# Orient the patterns with the same per-component sign as the scores (Step 5b),
# then rescale each subject's pattern to unit L2 norm: every participant panel of
# a component shares one colour limit, so without this a high-gain participant
# saturates the scale and a low-gain one looks flat. Correlations (topo_corr) are
# scale-invariant and therefore unaffected.
patterns_oriented = normalize_patterns_per_subject(
    iva_components * sign_per_comp[None, :, None]
)

topomap_root = PLOTS_DIR / "participant_topomaps"
if SAVE_PLOTS:
    written = plot_participant_topomaps(
        patterns_oriented,
        ref_topo,
        topo_corr,
        iva_info,
        n_channels,
        comp_indices,
        subject_ids,
        label=LABEL,
        root_dir=topomap_root,
    )
    print(f"Wrote {len(written)} figures ({len(subject_ids)} participants each).")
    print(f"Directory : {topomap_root}")
    for path in written[:3]:
        print(f"  {path.name}")
    if len(written) > 3:
        print(f"  ... and {len(written) - 3} more")
else:
    print("SAVE_PLOTS is False -> participant-comparison figures skipped.")